<a href="https://colab.research.google.com/github/sprothia/QWEN-Safety-Fine-Tuning/blob/main/FineTuning_Safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notes: You are not teaching new knowledge, not teaching math, not teaching reasoning, we are just adjusting behavior bias, that's why
we train for 1 epoch, not many, since it's such a small task

More epochs = stronger bias toward dataset distribution. Your dataset distribution = heavy refusal. So more epochs = heavy refusal everywhere.

In [ ]:
!pip install "huggingface-hub>=0.34.0,<1.0"
!pip install "datasets>=3.4.1,<4.4.0"

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
!pip uninstall -y bitsandbytes unsloth unsloth_zoo

!pip install --no-cache-dir bitsandbytes
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "Emilio407/Dolphin3.0-Qwen2.5-0.5B-GRPO-V1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
print("Loaded:", BASE_MODEL)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA attached.")

In [ ]:
from datasets import load_dataset

DATASET_NAME = "LucidityAI/Large-Safety-SFT"
ds = load_dataset(DATASET_NAME)
print(ds)
print(ds["train"][0])

In [ ]:
from datasets import DatasetDict
import torch

SYSTEM_PROMPT = (
    "You are a helpful assistant. You must refuse requests that involve wrongdoing, harm, or illegal activity. "
    "When refusing, be brief and offer a safer alternative when possible."
)

def format_example(ex):
    user = ex["question"]
    assistant = ex["answer"]

    if getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}

    text = (
        "<|im_start|>system\n" + SYSTEM_PROMPT + "<|im_end|>\n"
        "<|im_start|>user\n" + user + "<|im_end|>\n"
        "<|im_start|>assistant\n" + assistant + "<|im_end|>\n"
    )
    return {"text": text}

split = ds["train"].train_test_split(test_size=0.02, seed=42)
train_ds = split["train"].map(format_example, remove_columns=split["train"].column_names)
val_ds   = split["test"].map(format_example, remove_columns=split["test"].column_names)

print(train_ds[0]["text"][:400])
print("Train:", len(train_ds), "Val:", len(val_ds))

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

has_bf16 = torch.cuda.is_bf16_supported()

args = TrainingArguments(
    output_dir = "safety_sft_out",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,

    num_train_epochs = 1,
    learning_rate = 1e-4,
    warmup_ratio = 0.03,
    lr_scheduler_type = "cosine",
    logging_steps = 1,

    fp16 = not has_bf16,
    bf16 = has_bf16,

    optim = "paged_adamw_8bit",
    weight_decay = 0.0,
    max_grad_norm = 1.0,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = True,
    args = args,
)

trainer.train()

In [ ]:
model.save_pretrained("safety_model_lora")
tokenizer.save_pretrained("safety_model_lora")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("safety_model_lora", "/content/drive/MyDrive/safety_model_lora")